In [15]:
import pandas as pd

in_path  = "/workspace2/chiwu/works/critc_extraction/sentences_with_topic_new.xlsx"
out_path = "/workspace2/chiwu/works/critc_extraction/sentences_with_topic_merged.xlsx"

In [16]:
# 1) 读取
df = pd.read_excel(in_path)

In [17]:
# 2. 从 sheet (=topic_x) 提取数字
df["topic_num"] = df["sheet"].str.extract(r'(\d+)').astype(int)

# 3. 合并
merged = (
    df.groupby("themes", as_index=False)
      .agg(
          topic_idx = ("topic_num", "min"),                 # 最小序号
          sentences = ("sentences",  lambda s: "\n".join(s))
      )
)

# 4. 排序
merged = merged.sort_values("topic_idx")[["topic_idx", "themes", "sentences"]]

# 5. 生成所有 themes 的逗号串（去重后按 topic_idx 顺序）
all_themes_str = ",".join(merged["themes"].tolist())
print("所有主题串：")
print(all_themes_str)

所有主题串：
同业认可度,负类,销售趋势,偿债压力,上下游合作标准,涉诉记录,非银机构,行业风险,企业稳定性,经营数据,负债趋势,区域风险,负债结构,信用卡使用,流动资产周转率,交易数据,外部合作机构,关联企业跨行业,交易连续性,负债压力,房贷授信信息,交易起量时间,企业类型,审核标准与信息准确性,电核配合度,GMV差异,征信记录,征信白户风险,年龄与经营经验,还款/逾期行为,授信负债比,风险判定阈值,企业资质,新户风险,电核信息,网商贷风险,婚姻状态,还款行为,支用定价与存留期,上下游同一企业,销售同比数据缺失,网商贷生命周期风险


In [18]:
# 6. 写出到 Excel：Sheet1 为合并结果，Sheet2 保存主题串
with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
    merged.to_excel(writer, sheet_name="merged", index=False)
    # 单行两列：方便查看 / 复制
    pd.DataFrame({"all_themes":[all_themes_str]}).to_excel(
        writer, sheet_name="all_themes", index=False
    )

print(f"共 {len(merged)} 个主题，已保存到 {out_path}")


共 42 个主题，已保存到 /workspace2/chiwu/works/critc_extraction/sentences_with_topic_merged.xlsx
